In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import Compose, ToTensor
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import os
import librosa
import matplotlib.pyplot as plt


## Generating spectogram images with files used for reconstructing keystrokes

In [2]:
# waveform function for me to not bang my keyboard
def disp_waveform(signal, sr=None):
    plt.figure(figsize=(7,2))
    return librosa.display.waveshow(signal, sr=sr)

In [3]:
def isolator(signal, sample_rate, n_fft, hop_length, before, after, threshold=None, show=False,
             band_fmin=None, band_fmax=None):
    strokes = []
    stroke_times = []
    # -- signal'
    if show:
        disp_waveform(signal, sr=sample_rate)
    fft = librosa.stft(signal, n_fft=n_fft, hop_length=hop_length)

    # band-limited energy (optional)
    if band_fmin is not None or band_fmax is not None:
        freqs = librosa.fft_frequencies(sr=sample_rate, n_fft=n_fft)
        if band_fmin is None:
            band_fmin = 2000
        if band_fmax is None:
            band_fmax = 20000
        band_mask = (freqs >= band_fmin) & (freqs <= band_fmax)
        energy = np.sum(np.abs(fft[band_mask])**2, axis=0)
    else:
        energy = np.sum(np.abs(fft)**2, axis=0)   # power 합 (표준)

    # threshold auto from same signal unless provided
    if threshold is None:
        threshold = np.percentile(energy, threshold_percentile)
    # -- energy'
    if show:
        disp_waveform(energy)
    threshed = energy > threshold
    # -- peaks'
    if show:
        disp_waveform(threshed.astype(float))
    peaks = np.where(threshed == True)[0]
    peak_count = len(peaks)
    prev_end = sample_rate*min_keystroke_gap*(-1)
    # '-- isolating keystrokes'
    for i in range(peak_count):
        this_peak = peaks[i]
        timestamp = (this_peak*hop_length) + n_fft//2
        if timestamp > prev_end + (min_keystroke_gap*sample_rate):
            keystroke = signal[timestamp-before:timestamp+after]
            strokes.append(keystroke)
            stroke_times.append(timestamp / sample_rate)
            if show:
                disp_waveform(keystroke, sr=sample_rate)
            prev_end = timestamp+after
    print("peaks:", len(peaks))
    print("strokes:", len(strokes))
    
    return strokes, stroke_times


In [4]:
# Unified parameters (run this once)
n_fft = 1024
hop_length = 256
spec_n_fft = 1024
spec_hop_length = 225
spec_fmin = 0
spec_fmax = 19000
before = 2400
after = 5000

# threshold percentile used inside isolator when threshold=None
threshold_percentile = 90

# 최소 키 누르기 간격 (초 단위) - 동시 타이핑 고려
# for CoAtNet training dataset, set this to 0.5 ~ 1.0
# for GNN training dataset, set this to 0.03 ~ 0.1
min_keystroke_gap = 0.5  # 30ms, 필요시 0.03~0.1로 조정

# band-limited energy for keystroke detection
band_fmin = 2000
band_fmax = 20000


In [5]:
def create_dataset(n_fft, hop_length, before, after):
    for i, File in enumerate(keys):
        loc = MBP_AUDIO_DIR + File
        samples, sr = librosa.load(loc, sr=40000)
        # samples, sr = librosa.load(loc,sr=None,duration=1.0,mono=True)

        strokes = []
        thr = np.percentile(energy, 97) 
        step = 0.005
        strokes = isolator(samples[1*sr:], sr, n_fft, hop_length, before, after, thr, False, band_fmin=band_fmin, band_fmax=band_fmax )
        # while not len(strokes) == 25:
        #   strokes = isolator(samples[1*sr:], sr, n_fft, hop_length, before, after, prom, False )
        #   if len(strokes) < 25:
        #     prom -= step
        #   if len(strokes) > 25:
        #     prom += step
        #   if prom < 0:
        #     print("--not possible for : ", File)
        #     break
        #   step = step * 0.99
        label = [labels[i]]*len(strokes)
        data_dict['Key'] += label
        data_dict['File'] += strokes

    df = pd.DataFrame(data_dict)
    mapper = {}
    counter = 0
    for l in df['Key']:
        if not l in mapper:
            mapper[l] = counter
            counter += 1
    df.replace({'Key': mapper}, inplace = True)

    return df

## Generating Spectrograms


In [6]:
# --- Process wav/train files: strokes -> spectrogram images ---
from pathlib import Path
import matplotlib.pyplot as plt
import librosa.display

train_dir = Path('wav/train')
out_root = Path('out/train')
out_root.mkdir(parents=True, exist_ok=True)

for wav_path in sorted(train_dir.rglob('*.wav')):
    rel = wav_path.relative_to(train_dir)
    rel_dir = rel.parent
    key = wav_path.stem
    samples, sr = librosa.load(wav_path.as_posix(), sr=40000)

    print('file:', wav_path.name)
    print('sr:', sr, 'len:', len(samples))
    print('n_fft:', n_fft, 'hop:', hop_length, 'before:', before, 'after:', after)

    # isolate keystrokes with auto-threshold on the same signal
    strokes, stroke_times = isolator(samples, sr, n_fft, hop_length, before, after, threshold=None, show=False,
                                     band_fmin=band_fmin, band_fmax=band_fmax)

    # save spectrogram images
    key_out = out_root / rel_dir / key
    key_out.mkdir(parents=True, exist_ok=True)
    for i, stroke in enumerate(strokes):
        stft = librosa.stft(
            y=np.asarray(stroke).squeeze(),
            n_fft=spec_n_fft,
            hop_length=spec_hop_length,
        )
        spec = np.abs(stft) ** 2

        # optionally band-limit for display
        freqs = librosa.fft_frequencies(sr=sr, n_fft=spec_n_fft)
        if spec_fmin is not None or spec_fmax is not None:
            fmin = spec_fmin if spec_fmin is not None else freqs.min()
            fmax = spec_fmax if spec_fmax is not None else freqs.max()
            mask = (freqs >= fmin) & (freqs <= fmax)
            spec = spec[mask]

        plt.figure(figsize=(6, 4))
        librosa.display.specshow(
            librosa.power_to_db(spec, ref=np.max),
            x_axis='time',
            y_axis='linear',
            sr=sr,
            hop_length=spec_hop_length,
        )
        plt.colorbar(format='%+2.0f dB')
        plt.title(f'Spectrogram ({key}) - keystroke {i}')
        plt.tight_layout()
        plt.savefig(key_out / f'keystroke_{i:04d}.png', dpi=150)
        plt.close()


/Users/srsys_516/keyboard-sidechannel/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


file: 0.wav
sr: 40000 len: 3126614
n_fft: 1024 hop: 256 before: 2400 after: 5000
peaks: 1222
strokes: 100


/Users/srsys_516/keyboard-sidechannel/.venv/lib/python3.14/site-packages/numpy/_core/numeric.py:1211: RuntimeWarning: divide by zero encountered in dot
  res = dot(at, bt)
/Users/srsys_516/keyboard-sidechannel/.venv/lib/python3.14/site-packages/numpy/_core/numeric.py:1211: RuntimeWarning: overflow encountered in dot
  res = dot(at, bt)
/Users/srsys_516/keyboard-sidechannel/.venv/lib/python3.14/site-packages/numpy/_core/numeric.py:1211: RuntimeWarning: invalid value encountered in dot
  res = dot(at, bt)


file: 1.wav
sr: 40000 len: 2724694
n_fft: 1024 hop: 256 before: 2400 after: 5000
peaks: 1065
strokes: 98
file: 2.wav
sr: 40000 len: 2754560
n_fft: 1024 hop: 256 before: 2400 after: 5000
peaks: 1076
strokes: 99
file: 3.wav
sr: 40000 len: 2760534
n_fft: 1024 hop: 256 before: 2400 after: 5000
peaks: 1079
strokes: 101
file: 4.wav
sr: 40000 len: 2888534
n_fft: 1024 hop: 256 before: 2400 after: 5000
peaks: 1129
strokes: 100
file: 5.wav
sr: 40000 len: 2735787
n_fft: 1024 hop: 256 before: 2400 after: 5000
peaks: 1069
strokes: 97
file: 6.wav
sr: 40000 len: 2932054
n_fft: 1024 hop: 256 before: 2400 after: 5000
peaks: 1146
strokes: 100
file: 7.wav
sr: 40000 len: 2992640
n_fft: 1024 hop: 256 before: 2400 after: 5000
peaks: 1169
strokes: 98
file: 8.wav
sr: 40000 len: 3032747
n_fft: 1024 hop: 256 before: 2400 after: 5000
peaks: 1185
strokes: 100
file: 9.wav
sr: 40000 len: 2992640
n_fft: 1024 hop: 256 before: 2400 after: 5000
peaks: 1169
strokes: 100
file: Backspace.wav
sr: 40000 len: 3072000
n_fft: 